# Brax Ant — LWR-EGGROLL Experiment (Colab Runner)

1. Runtime > Change runtime type > **T4 GPU**
2. Run cells in order
3. Run keep-alive cell before leaving

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
WORK_DIR = '/content/drive/MyDrive/dissertation'
os.makedirs(WORK_DIR, exist_ok=True)
print(f'Work directory: {WORK_DIR}')

In [ ]:
# Cell 2: Verify GPU and install dependencies
!nvidia-smi
!pip install -q --upgrade "jax[cuda12]" 2>/dev/null || pip install -q --upgrade "jax[cuda11]"
!pip install -q brax==0.10.5 flax optax
import jax
print(f'JAX {jax.__version__}, devices: {jax.devices()}')

In [ ]:
# Cell 3: Clone repo and install HyperscaleES
import os
os.chdir('/content')
if not os.path.isdir('dissertation-repo'):
    !git clone https://github.com/Gagan0318/Dissertation---LWR-Eggroll.git dissertation-repo
else:
    os.chdir('dissertation-repo')
    !git pull
    os.chdir('/content')
os.chdir('dissertation-repo')
!git submodule update --init --recursive
!sed -i 's/requires-python = ">=3.11"/requires-python = ">=3.9"/' HyperscaleES/pyproject.toml 2>/dev/null
!sed -i 's/requires-python = ">=3.10"/requires-python = ">=3.9"/' HyperscaleES/pyproject.toml 2>/dev/null
!sed -i '/from.*llm/d; /from.*environments/d; /import.*reasoning_gym/d; /import.*tokenizers/d' HyperscaleES/src/hyperscalees/__init__.py 2>/dev/null
os.chdir('HyperscaleES')
!pip install -q -e .
os.chdir('/content/dissertation-repo')
from hyperscalees.noiser.eggroll import EggRoll
from hyperscalees.noiser.lwr_eggroll import LWREggRoll
from brax import envs
e = envs.get_environment('ant')
print(f'All OK — Ant obs={e.observation_size}, act={e.action_size}')

In [ ]:
# Cell 4: Restore cached results from cluster (if available)
import os, shutil
os.chdir('/content/dissertation-repo')
WORK_DIR = '/content/drive/MyDrive/dissertation'
TARBALL = os.path.join(WORK_DIR, 'results_brax_ant.tar.gz')
RESULTS_ON_DRIVE = os.path.join(WORK_DIR, 'results')
os.makedirs(os.path.join(RESULTS_ON_DRIVE, 'brax_ant', 'cache'), exist_ok=True)
if os.path.isfile(TARBALL):
    print('Found cluster cache tarball, unpacking...')
    !tar xzf {TARBALL} -C {WORK_DIR}
    print('Unpacked.')
else:
    print('No cluster tarball found — starting fresh or using existing Drive cache.')
local_results = '/content/dissertation-repo/results'
if os.path.islink(local_results):
    os.unlink(local_results)
elif os.path.isdir(local_results):
    local_brax = os.path.join(local_results, 'brax_ant')
    if os.path.isdir(local_brax):
        !cp -rn {local_brax}/* {RESULTS_ON_DRIVE}/brax_ant/ 2>/dev/null || true
    shutil.rmtree(local_results)
os.symlink(RESULTS_ON_DRIVE, local_results)
cache_dir = os.path.join(RESULTS_ON_DRIVE, 'brax_ant', 'cache')
cached = os.listdir(cache_dir) if os.path.isdir(cache_dir) else []
completed = [f for f in os.listdir(os.path.join(RESULTS_ON_DRIVE, 'brax_ant'))
             if f.endswith('.json') and f not in ('pilot_results.json', 'summary.json')] if os.path.isdir(os.path.join(RESULTS_ON_DRIVE, 'brax_ant')) else []
print(f'Cache files: {len(cached)}, Completed runs: {len(completed)}')
for f in sorted(completed): print(f'  {f}')

In [ ]:
# Cell 5: Run the experiment
import os, subprocess, threading
os.chdir('/content/dissertation-repo')
os.makedirs('logs', exist_ok=True)
def run_experiment():
    proc = subprocess.Popen(
        ['python3', '-u', 'experiments/brax_ant_experiment.py'],
        stdout=open('logs/brax_ant.log', 'w'),
        stderr=subprocess.STDOUT,
        cwd='/content/dissertation-repo')
    proc.wait()
    print(f'\nExperiment finished with exit code {proc.returncode}')
t = threading.Thread(target=run_experiment, daemon=True)
t.start()
print('Experiment launched. Run next cell to monitor, Cell 7 for keep-alive.')

In [ ]:
# Cell 6: Monitor progress (run anytime)
!tail -20 logs/brax_ant.log 2>/dev/null || echo 'No log yet'
print('---')
import os
rd = '/content/drive/MyDrive/dissertation/results/brax_ant'
cached = os.listdir(os.path.join(rd, 'cache')) if os.path.isdir(os.path.join(rd, 'cache')) else []
completed = [f for f in os.listdir(rd) if f.endswith('.json')
             and f not in ('pilot_results.json', 'summary.json')] if os.path.isdir(rd) else []
print(f'Cache: {len(cached)} files, Completed runs: {len(completed)}')

In [ ]:
# Cell 7: Keep-alive (run before leaving)
import time, threading
def keep_alive():
    while True:
        time.sleep(180)
        print('.', end='', flush=True)
ka = threading.Thread(target=keep_alive, daemon=True)
ka.start()
print('Keep-alive started. Leave this tab open.')

In [ ]:
# Cell 8: Save results to Drive (run when done or before disconnect)
import os
os.chdir('/content/dissertation-repo')
!tar czf /content/drive/MyDrive/dissertation/results_brax_ant.tar.gz results/brax_ant/
print('Results saved to Google Drive as results_brax_ant.tar.gz')